In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 28.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing ins

Loading the base model in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: LlamaTokenizer for unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit did not match tokenizer.json (transformers v5 replaced its pre-tokenizer), so it was rebuilt from tokenizer.json.
Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit as a legacy tokenizer.


Attach the LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.10.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Reasoning-style medical dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train[:1000]")
print(dataset[0].keys())   # Question, Complex_CoT, Response

README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json: reconstructing file:   0%|          |  0.00B / 58.2MB            

medical_o1_sft.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

dict_keys(['Question', 'Complex_CoT', 'Response'])


In [ ]:
train_prompt_style = """Below is a medical question. Think carefully, then answer.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

EOS_TOKEN = tokenizer.eos_token

def format_row(examples):
    texts = []
    for q, cot, ans in zip(examples["Question"], examples["Complex_CoT"], examples["Response"]):
        texts.append(train_prompt_style.format(q, cot, ans) + EOS_TOKEN)
    return {"text": texts}

dataset = dataset.map(format_row, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        warmup_steps = 5,
        logging_steps = 10,
        output_dir = "outputs",
        seed = 3407,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
gpu = torch.cuda.get_device_properties(0)
start = round(torch.cuda.max_memory_reserved()/1024**3, 3)
print(f"{gpu.name}, total {round(gpu.total_memory/1024**3,3)} GB, reserved before training: {start} GB")

stats = trainer.train()

used = round(torch.cuda.max_memory_reserved()/1024**3, 3)
print(f"Training time: {round(stats.metrics['train_runtime']/60,2)} min")
print(f"Peak memory: {used} GB, used for training: {round(used-start,3)} GB")

Tesla T4, total 14.563 GB, reserved before training: 5.787 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.932636
20,1.568231
30,1.543673
40,1.497935
50,1.536551
60,1.465028
70,1.500157
80,1.468312
90,1.483230
100,1.469580


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-125/tokenizer_config.json.


Training time: 35.73 min
Peak memory: 7.184 GB, used for training: 1.397 GB


In [ ]:
model.save_pretrained("medical_lora_adapter")
tokenizer.save_pretrained("medical_lora_adapter")

Unsloth: Restored added_tokens_decoder metadata in medical_lora_adapter/tokenizer_config.json.


('medical_lora_adapter/tokenizer_config.json',
 'medical_lora_adapter/chat_template.jinja',
 'medical_lora_adapter/tokenizer.json')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("medical_lora_adapter", "zip", "medical_lora_adapter")

files.download("medical_lora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
FastLanguageModel.for_inference(model)

prompt = train_prompt_style.format(
    "A 45-year-old man presents with chest pain radiating to the left arm. What are the most likely causes and first steps in management?",
    "",   # empty so the model writes its own reasoning
    ""
)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=600, temperature=0.6, use_cache=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is a medical question. Think carefully, then answer.

### Question:
A 45-year-old man presents with chest pain radiating to the left arm. What are the most likely causes and first steps in management?

### Response:
<think>

</think>
Alright, let's figure out what's going on with this guy. He's a 45-year-old who's having chest pain that's radiating to his left arm. That's definitely a red flag because it sounds like angina, but we can't be certain yet.

First off, angina is usually caused by reduced blood flow to the heart muscle. This could be due to atherosclerosis, where plaque builds up in the arteries. Now, there are a few types of angina, and it's important to figure out which one we're dealing with.

If it's just about the usual chest pain and nothing unusual, it's likely to be stable angina. In that case, we’d want to think about his lifestyle, like diet and exercise. But if the pain feels worse with exertion or is more severe, we should consider unstable angina or someth

In [ ]:
inference_prompt = """Below is a medical question. Think carefully, then answer.

### Question:
{}

### Response:
<think>
"""

def ask(question, use_adapter=True, max_new_tokens=700):
    FastLanguageModel.for_inference(model)
    inputs = tokenizer([inference_prompt.format(question)], return_tensors="pt").to("cuda")
    if use_adapter:
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.6, use_cache=True)
    else:
        with model.disable_adapter():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.6, use_cache=True)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("### Response:")[-1]

test_questions = [
    "A 45-year-old man presents with chest pain radiating to the left arm. What are the most likely causes and first steps in management?",
    "A 30-year-old woman has fatigue, weight gain, cold intolerance and constipation. What is the likely diagnosis and which test confirms it?",
    "A 6-year-old child has fever, sore throat and a sandpaper-like rash. What is the diagnosis and treatment?",
    "A diabetic patient is brought in confused with fruity-smelling breath and rapid deep breathing. What is the diagnosis and initial management?",
]

for q in test_questions:
    print("QUESTION:", q)
    print("\n--- BASE MODEL ---\n", ask(q, use_adapter=False))
    print("\n--- FINE-TUNED v1 ---\n", ask(q, use_adapter=True))
    print("\n" + "="*80 + "\n")

QUESTION: A 45-year-old man presents with chest pain radiating to the left arm. What are the most likely causes and first steps in management?


Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- BASE MODEL ---
 
<think>
Alright, so I have this medical question here: A 45-year-old man presents with chest pain radiating to the left arm. What are the most likely causes and first steps in management?

Okay, let me break this down. The patient is a 45-year-old man with chest pain that's radiating to his left arm. That makes me think of a few possibilities. First, I remember that chest pain can be caused by a variety of issues, ranging from something as simple as indigestion to something more serious like a heart attack.

The fact that the pain is radiating to the left arm makes me think about angina or a myocardial infarction (heart attack). I recall that the discomfort from a heart attack often starts in the chest and spreads to the arm, jaw, or back. It's usually described as a squeezing or pressure in the chest.

But wait, could it be something else? Maybe a panic attack or anxiety? I know that anxiety can cause physical symptoms like chest pain, but the radiation to the ar

Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- FINE-TUNED v1 ---
 
<think>
Okay, so I'm trying to figure out what the most likely causes are for a 45-year-old man who presents with chest pain that's radiating to his left arm. And also, what the first steps in management would be. Hmm, let me think through this step by step.

First off, chest pain can be caused by a lot of different things, but when it's radiating to the left arm, that makes me think of specific conditions. I remember from my studies that chest pain with radiation to an arm, particularly the left one, is often associated with a heart issue. The most common one that comes to mind is a myocardial infarction, or heart attack. That makes sense because the pain from a heart attack can spread to the arm, and it's usually on the left side since that's where the left coronary artery is.

But wait, could it be something else? Maybe a pulmonary issue? I know that sometimes chest pain can be related to the lungs, like a pulmonary embolism, but those usually present with s

...


--- BASE MODEL ---
 
<think>
Okay, so I've got a 6-year-old child with a fever, sore throat, and a rash that feels like sandpaper. Hmm, I need to figure out what's going on here. Let me think through this step by step.

First, the child has a fever, which is a common symptom in many illnesses. Sore throat too—could be from a lot of things, maybe viral or bacterial. The rash is key here. It's described as sandpaper-like, which makes me think of something that's dry and scaly. I've heard that in measles, the rash comes in stages and can be quite itchy. But wait, measles is pretty rare these days, right? Or is it making a comeback? Maybe, but I should consider other possibilities.

I remember that chickenpox is caused by herpes virus and also has a rash, but usually it's more vesicular, like blisters. This is more like dry scales, so maybe not chickenpox. Then there's eczema, but that's usually more red and inflamed, not dry. Maybe psoriasis? But that's more in adults, though some kids c

Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- FINE-TUNED v1 ---
 
<think>
Okay, so I'm trying to figure out what's going on with this 6-year-old child who has a fever, sore throat, and a rash that feels like sandpaper. Hmm, let's break this down step by step.

First, the fever and sore throat are pretty common in kids, but the rash is a big clue. The rash being described as sandpaper-like makes me think of something that's dry and rough. I remember that chickenpox, or varicella, has a distinctive rash. Chickenpox typically starts with an itchy, vesicular rash that can be scaly or dry, especially as it heals. The fever and sore throat also fit with chickenpox because it's a common presentation.

But wait, could it be something else? I should consider other possibilities. For example, measles also has a rash, but it's more of a red, raised, and edematous rash, not dry and scaly. Plus, measles is less common these days because of vaccination. Then there's meningitis, which can cause a rash, but it's usually more widespread and a

Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- BASE MODEL ---
 
<think>
Okay, so I have a medical question here about a diabetic patient who's brought in confused with fruity-smelling breath and rapid deep breathing. I need to figure out the diagnosis and initial management. Let me think through this step by step.

First, I know that diabetes can lead to various complications. The patient is diabetic, so that's a given. Now, the key symptoms here are confusion, fruity breath, and rapid deep breathing. I remember that in diabetic patients, ketoacidosis is a common and serious condition. 

Wait, what's ketoacidosis? I think it's when the body produces excess ketones because it's not getting enough insulin. High ketone levels can make the breath smell fruity. That fits with the patient's symptoms. So, this is likely diabetic ketoacidosis (DKA).

But why is the patient confused? I recall that high blood glucose levels can cause hypoglycemia, which is low blood sugar. Hypoglycemia can lead to confusion, rapid breathing, and other s

In [ ]:
more_data = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train[1000:3000]")
more_data = more_data.map(format_row, batched=True)
print(len(more_data))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

2000


In [ ]:
FastLanguageModel.for_training(model)   # important: switches back from inference mode

from trl import SFTTrainer, SFTConfig

trainer2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = more_data,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        warmup_steps = 5,
        logging_steps = 10,
        save_steps = 50,
        save_total_limit = 2,
        output_dir = "outputs_v2",
        report_to = "none",
        seed = 3407,
    ),
)

start = round(torch.cuda.max_memory_reserved()/1024**3, 3)
stats2 = trainer2.train()
used = round(torch.cuda.max_memory_reserved()/1024**3, 3)
print(f"Training time: {round(stats2.metrics['train_runtime']/60,2)} min")
print(f"Peak memory: {used} GB")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.014234
20,1.703103
30,1.584040
40,1.564181
50,1.521708
60,1.474014
70,1.504886
80,1.483140
90,1.509305
100,1.454136


Unsloth: Restored added_tokens_decoder metadata in outputs_v2/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v2/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v2/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v2/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v2/checkpoint-250/tokenizer_config.json.


Training time: 67.84 min
Peak memory: 7.168 GB


In [ ]:
for q in test_questions:
    print("QUESTION:", q)
    print("\n--- FINE-TUNED v2 ---\n", ask(q, use_adapter=True))
    print("\n" + "="*80 + "\n")

Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: A 45-year-old man presents with chest pain radiating to the left arm. What are the most likely causes and first steps in management?


Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- FINE-TUNED v2 ---
 
<think>
Alright, let's think about this. A 45-year-old guy comes in with chest pain, and it's radiating to his left arm. Hmm, that sounds like a classic heart attack warning. But I need to be careful here. It could also be something else, like a heart attack itself or maybe a gastrointestinal issue like a heartburn or something. But let's focus on heart attack first.

Now, I know that heart attacks are usually caused by a blockage in a coronary artery. There are a few scenarios here: atherosclerosis, which is the most common, or maybe something like a coronary artery spasm, or maybe even something like an aortic dissection. But let's stick to what's common.

So, what should I do to manage this? First, I need to make sure he's not having a heart attack. That means I should check his heart rate and blood pressure. If he's really having a heart attack, he'll have chest pain, shortness of breath, and maybe a cold, clammy skin. That's pretty characteristic. 

Next, 

Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- FINE-TUNED v2 ---
 
<think>
Alright, so we've got a 30-year-old woman experiencing fatigue, gaining weight, feeling cold, and having constipation. Hmm, that sounds like a lot of different symptoms. Let's think about what could be going on here.

First, fatigue and weight gain can happen in a bunch of conditions, but cold intolerance and constipation make me think of something else. These symptoms could be pointing toward hypothyroidism, but wait, hypothyroidism usually doesn't cause constipation. So maybe it's something else.

Oh, maybe it's hypogonadism? That's when the body doesn't produce enough hormones, like estrogen or testosterone, which can definitely cause these symptoms. But why would she have constipation?

Let's consider hyperprolactinemia. It's when there's too much prolactin in the body, which can mess with the body's hormones and lead to these symptoms. It fits pretty well with what she's experiencing: fatigue, weight gain, cold intolerance, and constipation.

Okay,

Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- FINE-TUNED v2 ---
 
<think>
Okay, let's think about this. A 6-year-old with a fever, sore throat, and a rash that feels like sandpaper. That's definitely making me think of something like measles. It's known for those kinds of symptoms.

Measles is a viral infection, and it's pretty distinctive with that rash. It's usually the first thing you think of when you see a child with these symptoms. I remember reading that measles is a classic example where the rash appears in stages, starting from the face and neck and then spreading down the body.

Now, when it comes to treatment, measles is pretty tricky. There isn't really a specific drug that can cure it. The best thing we can do is support the child while the virus works its way through them. That means lots of fluids, good nutrition, and keeping them comfortable.

I should also consider the measles vaccine. It's super effective at preventing the disease altogether. But since this child is already showing symptoms, the vaccine won'

In [ ]:
import shutil
from google.colab import files

model.save_pretrained("medical_lora_adapter_v2")
tokenizer.save_pretrained("medical_lora_adapter_v2")

shutil.make_archive("medical_lora_adapter_v2", "zip", "medical_lora_adapter_v2")
files.download("medical_lora_adapter_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
eval_set = [
    ("A 45-year-old man presents with chest pain radiating to the left arm. What are the most likely causes and first steps in management?",
     [["ecg","electrocardiogram"],["aspirin"],["troponin"]]),
    ("A 30-year-old woman has fatigue, weight gain, cold intolerance and constipation. What is the likely diagnosis and which test confirms it?",
     [["hypothyroid"],["tsh"]]),
    ("A 6-year-old child has fever, sore throat and a sandpaper-like rash. What is the diagnosis and treatment?",
     [["scarlet fever","group a strep","streptococc"],["penicillin","amoxicillin"]]),
    ("A diabetic patient is brought in confused with fruity-smelling breath and rapid deep breathing. What is the diagnosis and initial management?",
     [["ketoacidosis","dka"],["insulin"],["fluid","saline"]]),
    ("A 25-year-old woman develops sudden shortness of breath and pleuritic chest pain after a long flight. What is the most likely diagnosis and which test confirms it?",
     [["pulmonary embolism"],["angiogra","d-dimer"]]),
    ("A 60-year-old man has sudden facial droop, arm weakness and slurred speech that started one hour ago. What is the diagnosis and first investigation and treatment?",
     [["stroke"],["non-contrast","ct scan","ct head","head ct","computed tomography"],["alteplase","thrombolysis","thrombolytic","tpa"]]),
    ("A 22-year-old has fever, nausea and pain that started around the umbilicus and moved to the right lower abdomen. What is the diagnosis and management?",
     [["appendicitis"],["appendectomy","appendicectomy","surgery","surgical"]]),
    ("A child has a barking cough, hoarseness and inspiratory stridor. What is the diagnosis and treatment?",
     [["croup","laryngotracheobronchitis"],["dexamethasone","steroid","corticosteroid"]]),
]

def generate(question, use_adapter, max_new_tokens=1500):
    FastLanguageModel.for_inference(model)
    inputs = tokenizer([inference_prompt.format(question)], return_tensors="pt").to("cuda")
    if use_adapter:
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    else:
        with model.disable_adapter():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    text = tokenizer.decode(out[0], skip_special_tokens=True).split("### Response:")[-1]
    return text

def score(text, groups):
    answer = text.split("</think>")[-1].lower() if "</think>" in text else ""
    finished = "</think>" in text
    hits = sum(any(k in answer for k in g) for g in groups)
    return finished, hits, len(groups)

results = {}
for name, use_adapter in [("base", False), ("v2", True)]:
    tot_hits = tot_groups = fin = 0
    for q, groups in eval_set:
        f, h, n = score(generate(q, use_adapter), groups)
        fin += f; tot_hits += h; tot_groups += n
    results[name] = (fin, tot_hits, tot_groups)
    print(f"{name}: finished {fin}/{len(eval_set)} answers, checklist {tot_hits}/{tot_groups}")

Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t

base: finished 8/8 answers, checklist 14/19


Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t

v2: finished 8/8 answers, checklist 11/19


In [ ]:
import time

def run_eval(use_adapter):
    rows = []
    for q, groups in eval_set:
        t = time.time()
        text = generate(q, use_adapter)
        n_tokens = len(tokenizer(text)["input_ids"])
        f, h, n = score(text, groups)
        rows.append((q[:60], h, n, n_tokens, round(time.time()-t)))
    return rows

for name, flag in [("base", False), ("v2", True)]:
    rows = run_eval(flag)
    print(f"\n{name}")
    for r in rows:
        print(f"  hits {r[1]}/{r[2]} | tokens {r[3]} | {r[4]}s | {r[0]}")
    print(f"  avg tokens {sum(r[3] for r in rows)/len(rows):.0f}, checklist {sum(r[1] for r in rows)}/{sum(r[2] for r in rows)}")

Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t


base
  hits 2/3 | tokens 844 | 39s | A 45-year-old man presents with chest pain radiating to the 
  hits 2/2 | tokens 884 | 44s | A 30-year-old woman has fatigue, weight gain, cold intoleran
  hits 0/2 | tokens 1068 | 55s | A 6-year-old child has fever, sore throat and a sandpaper-li
  hits 3/3 | tokens 852 | 44s | A diabetic patient is brought in confused with fruity-smelli
  hits 1/2 | tokens 713 | 35s | A 25-year-old woman develops sudden shortness of breath and 
  hits 3/3 | tokens 1008 | 52s | A 60-year-old man has sudden facial droop, arm weakness and 
  hits 2/2 | tokens 939 | 47s | A 22-year-old has fever, nausea and pain that started around
  hits 1/2 | tokens 814 | 41s | A child has a barking cough, hoarseness and inspiratory stri
  avg tokens 890, checklist 14/19


Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/t


v2
  hits 1/3 | tokens 617 | 39s | A 45-year-old man presents with chest pain radiating to the 
  hits 0/2 | tokens 480 | 30s | A 30-year-old woman has fatigue, weight gain, cold intoleran
  hits 0/2 | tokens 762 | 48s | A 6-year-old child has fever, sore throat and a sandpaper-li
  hits 2/3 | tokens 567 | 35s | A diabetic patient is brought in confused with fruity-smelli
  hits 2/2 | tokens 533 | 33s | A 25-year-old woman develops sudden shortness of breath and 
  hits 3/3 | tokens 587 | 38s | A 60-year-old man has sudden facial droop, arm weakness and 
  hits 2/2 | tokens 663 | 43s | A 22-year-old has fever, nausea and pain that started around
  hits 1/2 | tokens 598 | 38s | A child has a barking cough, hoarseness and inspiratory stri
  avg tokens 601, checklist 11/19


In [ ]:
import gc
for n in ["model", "trainer", "trainer2"]:
    globals().pop(n, None)      # trainer may not exist in this session
gc.collect(); torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

from trl import SFTTrainer, SFTConfig
trainer3 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = more_data,          # same 2000 rows as v2, so the comparison is fair
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 5e-5,           # the only change from v2
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        warmup_steps = 5,
        logging_steps = 10,
        save_steps = 50,
        save_total_limit = 2,
        output_dir = "outputs_v3",
        report_to = "none",
        seed = 3407,
    ),
)
stats3 = trainer3.train()
print(f"Training time: {round(stats3.metrics['train_runtime']/60,2)} min")
print(f"Peak memory: {round(torch.cuda.max_memory_reserved()/1024**3, 3)} GB")

==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit as a legacy tokenizer.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.074379
20,1.842721
30,1.660645
40,1.616606
50,1.568002
60,1.521112
70,1.546035
80,1.520554
90,1.541829
100,1.481032


Unsloth: Restored added_tokens_decoder metadata in outputs_v3/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v3/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v3/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v3/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_v3/checkpoint-250/tokenizer_config.json.


Training time: 63.49 min
Peak memory: 7.203 GB


In [ ]:
rows = run_eval(True)
print("\nv3 (lr 5e-5)")
for r in rows:
    print(f"  hits {r[1]}/{r[2]} | tokens {r[3]} | {r[4]}s | {r[0]}")
print(f"  avg tokens {sum(r[3] for r in rows)/len(rows):.0f}, checklist {sum(r[1] for r in rows)}/{sum(r[2] for r in rows)}")

model.save_pretrained("medical_lora_adapter_v3")
tokenizer.save_pretrained("medical_lora_adapter_v3")
shutil.make_archive("medical_lora_adapter_v3", "zip", "medical_lora_adapter_v3")
files.download("medical_lora_adapter_v3.zip")